In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

# -------------------------------
# Step 0: Setup idealized stratification
# -------------------------------
H = 5000.0
nz = 501
nt = 300             # number of time steps
z = np.linspace(0, -H, nz)
t = np.arange(nt)    # arbitrary time units

# Exponential N² profile
N0 = 5e-3
h = 1000.0
N2 = N0**2 * np.exp(z / h)   # shape (nz,)

# -------------------------------
# Step 1: Vectorized vertical mode solver
# -------------------------------
def vertical_modes_vectorized(N2, z, n_modes):
    nz = len(z)
    dz = z[1] - z[0]
    invN2 = 1.0 / N2
    
    lower = (invN2[:-1] + invN2[1:]) / dz**2
    upper = lower.copy()
    diag = -np.concatenate(([0], invN2[:-2] + 2*invN2[1:-1] + invN2[2:], [0])) / dz**2

    # BCs
    diag[0] = 1.0e12
    diag[-1] = -1.0 / dz
    lower[-1] = 1.0 / dz

    L = np.diag(diag) + np.diag(lower, -1) + np.diag(upper, 1)
    eigvals, eigvecs = la.eig(L)
    eigvals, eigvecs = eigvals.real, eigvecs.real

    idx = np.argsort(eigvals)
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]

    neg_idx = np.where(eigvals < 0)[0][:n_modes]
    eigvals_sel = eigvals[neg_idx]
    phi = eigvecs[:, neg_idx].T

    norms = np.sqrt(np.trapz(phi**2 * N2, z, axis=1))
    phi /= norms[:, np.newaxis]

    c = 1.0 / np.sqrt(-eigvals_sel)
    return phi, c

n_modes = 3
phi, c = vertical_modes_vectorized(N2, z, n_modes)
print("Modal speeds (m/s):", c)

# -------------------------------
# Step 2: Create synthetic v'(z,t)
# -------------------------------
# Combine modes with different time-dependent amplitudes
np.random.seed(0)
amp_time = np.sin(2*np.pi*t[:, np.newaxis]/50)  # period ~50 steps
amp_time2 = 0.5*np.cos(2*np.pi*t[:, np.newaxis]/80)
amp_time3 = 0.2*np.sin(2*np.pi*t[:, np.newaxis]/30)

# Stack amplitudes as (nt, n_modes)
A_modes_time = np.hstack([amp_time, amp_time2, amp_time3])  # shape (nt, n_modes)

# Project into vertical profiles: v'(z,t) = A_modes_time @ phi
v_prime = A_modes_time @ phi      # shape (nt, nz)

# -------------------------------
# Step 3: Project v'(z,t) onto modes (vectorized)
# -------------------------------
# Weighted integral over z using N2
# A_n(t) = ∫ phi_n(z) * v'(z,t) * N2 dz
A_n_recovered = np.trapz(phi[np.newaxis, :, :] * v_prime[:, np.newaxis, :] * N2[np.newaxis, np.newaxis, :], axis=2)
# shape (nt, n_modes)
print("Recovered shape:", A_n_recovered.shape)

# -------------------------------
# Step 4: Plot modal time series
# -------------------------------
plt.figure(figsize=(8,5))
for n in range(n_modes):
    plt.plot(t, A_n_recovered[:,n], label=f"Mode {n+1}")
plt.xlabel("Time step")
plt.ylabel("Modal amplitude")
plt.title("Time series of vertical modal amplitudes")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# -------------------------------
# Step 5: Compare original and reconstructed v'(z,t) at a time slice
# -------------------------------
t_idx = 100
v_recon = (A_n_recovered[t_idx][:, np.newaxis] * phi).sum(axis=0)

plt.figure(figsize=(6,5))
plt.plot(v_prime[t_idx], z, 'k', label="Original v'(t=100)")
plt.plot(v_recon, z, 'r--', label="Reconstruction")
plt.xlabel("v'")
plt.ylabel("Depth (m)")
plt.gca().invert_yaxis()
plt.legend()
plt.title("Reconstruction at t=100")
plt.grid()
plt.show()